In [21]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
llm1 = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation"
)

llm2 = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    task="text-generation"
)



In [23]:
generator = ChatHuggingFace(llm=llm1)
evaluator = ChatHuggingFace(llm=llm2)
optimizer = ChatHuggingFace(llm=llm1)

In [24]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate


class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(
        description="Final evaluation of the tweet"
    )
    feedback: str = Field(...,
        description="Detailed feedback on the tweet, including what was done well and what could be improved"
    )

# Use JsonOutputParser instead of with_structured_output
parser = JsonOutputParser(pydantic_object=TweetEvaluation)

# Create a prompt that instructs the model to output JSON
prompt = PromptTemplate(
    template="Analyze the Quality of the following tweet and respond in JSON format with 'evaluation' and 'feedback' fields.\n\nTweet: {tweet}\n\n{format_instructions}",
    input_variables=["tweet"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# Chain the model with the parser
structured_eva_model = prompt | evaluator | parser

In [25]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation : Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

In [26]:
def generate_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a social media manager. Your task is to create a tweet based on the given topic."),
        HumanMessage(content=f"Topic: {state['topic']}")
    ]
    response = generator.invoke(messages).content

    return {
        'tweet': response
    }



def evaluate_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a senior social media manager. Your task is to evaluate the quality of the given tweet and provide feedback for improvement if necessary."),
        HumanMessage(content=f"Tweet: {state['tweet']}")
    ]
    response = structured_eva_model.invoke(messages)

    # For simplicity, we assume that the response contains both the evaluation and feedback in a structured format.
    # In a real implementation, you would need to parse the response accordingly.

    return {
        'evaluation': response['evaluation'],
        'feedback': response['feedback']
    }


def optimize_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a social media manager. Your task is to optimize the given tweet based on the provided feedback."),
        HumanMessage(content=f"Current Tweet: {state['tweet']}\nFeedback: {state['feedback']}")
    ]
    response = optimizer.invoke(messages).content
    iteration = state['iteration'] + 1

    return {
        'tweet': response,
        'iteration': iteration
    }

def route_evaluation(state: TweetState):
    if state['evaluation'] == 'approved':
        return 'approved'
    elif state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [27]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)


graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate',route_evaluation, { 'approved': END, 'needs_improvement': 'optimize' })
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

In [28]:
initial_state = {
    'topic': "The importance of AI in modern society",
    'tweet': "",
    'evaluation': "",
    'feedback': "",
    'iteration': 0,
    'max_iteration': 3
}
result = workflow.invoke(initial_state)
print(result)

{'topic': 'The importance of AI in modern society', 'tweet': 'AI isn’t just tech—it’s the backbone of modern life. From healthcare diagnostics to climate modeling, AI powers smarter decisions, fuels innovation, and bridges gaps we once thought impossible. 🌍🤖 #AI #FutureNow #TechForGood', 'evaluation': 'approved', 'feedback': "Great use of emojis and relevant hashtags to boost visibility. The message is clear, inspiring, and highlights AI's transformative role across key sectors. To increase engagement, consider adding a question or call-to-action to spark conversation.", 'iteration': 0, 'max_iteration': 3}
